## Item-Based Collaborative Filtering (IBCF)
- Memory-based recommendation approach
- Identifies similar items based on user ratings

### 1. Import all libraries

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import time
from math import sqrt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

### 2. Load Dataset

In [2]:
# Import data of users
user_df = pd.read_csv('C:/Users/nakam/Documents/TDA6323 - ALGORITHM DESIGN AND ANALYSIS/Project Guideline/Collaborative-Filtering-Algorithms-in-Recommendation-Systems/datasets/ml-100k/u.data', 
                        sep="\t",
                        header=None,
                        usecols=[0, 1, 2],
                        names=["user_id", "movie_id", "rating"]
                        )

# Import data of items
item_df = pd.read_csv('C:/Users/nakam/Documents/TDA6323 - ALGORITHM DESIGN AND ANALYSIS/Project Guideline/Collaborative-Filtering-Algorithms-in-Recommendation-Systems/datasets/ml-100k/u.item',
                        sep="|",
                        header=None,
                        encoding="latin-1",
                        names=[
                            "movie_id", "title", "release_date", "video_release_date",
                            "IMDb_URL", "unknown", "Action", "Adventure", "Animation",
                            "Children's", "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
                            "Film-Noir", "Horror", "Musical", "Mystery", "Romance", "Sci-Fi",
                            "Thriller", "War", "Western"
                        ])

In [3]:
# Use 100k ratings for experiment
user_df = user_df.head(100000)

print(f"Ratings dataset shape: {user_df.shape}")

Ratings dataset shape: (100000, 3)


In [4]:
item_df.head()

,movie_id,title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


### 3. Build User-Item Matrix
- Rows = users
- Columns = movies
- Values = ratings


In [5]:
# Build user-item matrix
user_item_matrix = user_df.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

print(f"Matrix shape: {user_item_matrix.shape}")
print(f"Matrix size: {user_item_matrix.size}\n")
print(user_item_matrix.head())

Matrix shape: (943, 1682)
Matrix size: 1586126

movie_id  1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                               ...   
1          5.0   3.0   4.0   3.0   3.0   5.0   4.0   1.0   5.0   3.0  ...   
2          4.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   2.0  ...   
3          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
4          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
5          4.0   3.0   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   

movie_id  1673  1674  1675  1676  1677  1678  1679  1680  1681  1682  
user_id                                                               
1          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
2          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
3          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  
4          NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN 

In [6]:
# Fill missing ratings with 0 for unrated items
user_item_matrix = user_item_matrix.fillna(0)

In [7]:
# Re-check the matrix after filling missing values
print(user_item_matrix.head())

movie_id  1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                               ...   
1          5.0   3.0   4.0   3.0   3.0   5.0   4.0   1.0   5.0   3.0  ...   
2          4.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   2.0  ...   
3          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
4          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
5          4.0   3.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   

movie_id  1673  1674  1675  1676  1677  1678  1679  1680  1681  1682  
user_id                                                               
1          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
2          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
3          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
4          0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  
5          0.0   0.0   0.0   0.0  

### 4. Compute Item similarity
Use Cosine similarity to calculate similarity between each movie pair.

In [8]:
# Transpose to get rows = items, column, users
item_matrix = user_item_matrix.T

# Compute Item-Item Cosine similarity between every pair of items
item_similarity = cosine_similarity(item_matrix)

### 5. Item similarity Matrix

In [9]:
# Wrap in a DataFrame for better visualization
item_similarity_df = pd.DataFrame(
    item_similarity,
    index = user_item_matrix.columns,
    columns = user_item_matrix.columns
).round(4)

print(f"Item-Item Similarity Matrix shape: {item_similarity_df.shape}\n")
print(item_similarity_df.head())

Item-Item Similarity Matrix shape: (1682, 1682)

movie_id    1       2       3       4       5       6       7       8     \
movie_id                                                                   
1         1.0000  0.4024  0.3302  0.4549  0.2867  0.1163  0.6210  0.4811   
2         0.4024  1.0000  0.2731  0.5026  0.3188  0.0836  0.3834  0.3370   
3         0.3302  0.2731  1.0000  0.3249  0.2130  0.1067  0.3729  0.2008   
4         0.4549  0.5026  0.3249  1.0000  0.3342  0.0903  0.4893  0.4902   
5         0.2867  0.3188  0.2130  0.3342  1.0000  0.0373  0.3348  0.2592   

movie_id    9       10    ...    1673  1674   1675   1676    1677  1678  1679  \
movie_id                  ...                                                   
1         0.4963  0.2739  ...  0.0354   0.0  0.000  0.000  0.0354   0.0   0.0   
2         0.2553  0.1711  ...  0.0000   0.0  0.000  0.000  0.0000   0.0   0.0   
3         0.2737  0.1581  ...  0.0000   0.0  0.000  0.000  0.0323   0.0   0.0   
4         0.4

### 6. Select Target User
Find movies that the user has already rated.

In [10]:
# Change user_id to test different users
target_user = user_item_matrix.index[0] # Example: user_id = 1

# Get the ratings of the target user
rated_movies = user_item_matrix.loc[target_user]
rated_movies = rated_movies[rated_movies > 0] # Rated movies only

# Unrated movies by the target user - Potential movies for recommendation
unrated_movies = user_item_matrix.loc[target_user]
unrated_movies = unrated_movies[unrated_movies == 0] # Unrated movies only

print(f"Rated movies by User {target_user}: {len(rated_movies)}")
print(f"Unrated movies by User {target_user}: {len(unrated_movies)}")


Rated movies by User 1: 272
Unrated movies by User 1: 1410


### 7. Find top-N Similar Items
Use Rated Neighbours to compute score for each unrated movie.

In [11]:
N = 10 # Number of top similar items to consider for prediction

# Function: Predict rating for an unrated movie using weighted average of N most similar rated movies
def predict_ratings(user_id, movie_id, user_item_matrix, item_similarity_df, N):
    # Ratings of the target user
    user_ratings = user_item_matrix.loc[user_id]
    user_ratings = user_ratings[user_ratings > 0] # Rated movies only

    # Similarity scores between target movie with all other rated movies
    similarity_scores = item_similarity_df[movie_id][user_ratings.index]

    # Choose top-N most similar rated movies
    top_n = similarity_scores.sort_values(ascending=False).head(N)

    # Return weighted average of ratings if no similar movies are rated
    if top_n.sum() == 0:
        return user_ratings.mean() # Fallback to user's average rating
    
    # Calculate weighted average rating
    numerator = sum(top_n[movie] * user_ratings[movie] for movie in top_n.index)
    denominator = top_n.sum()
    
    return numerator / denominator

# Generate predicted scores for all unrated movies for the target user
predicted_scores = {}

for movie_id in unrated_movies:
    for movie_id in unrated_movies.index:
        predicted_scores[movie_id] = predict_ratings(
            target_user,
            movie_id,
            user_item_matrix,
            item_similarity_df,
            N
        )

### 8. Rank and Recommend Item
Return top-K movies with highest predicted scores.

In [12]:
# Number of movies to recommend
K = 10

# Sort predicted scores and return top-K recommended movies
def recommend_movies(predictions, item_df, K):
    # Sort predicted scores in descending order
    top_k = dict(sorted(predictions.items(), key=lambda item: item[1], reverse=True))

    results = []
    for item_id, score in top_k.items():
        title = item_df.loc[item_df['movie_id'] == item_id, 'title'].values
        title = title[0] if len(title) > 0 else "Unknown"
        results.append({
            "movie_id": movie_id,
            "title": title,
            "predicted_rating": round(score, 4)
        })
    return pd.DataFrame(results)

recommendations = recommend_movies(predicted_scores, item_df, K)
print(f"Top-{K} Recommended movies for User {target_user}:\n")
print(recommendations.head(K))


Top-10 Recommended movies for User 1:

   movie_id                                              title  \
0      1682                    Everyone Says I Love You (1996)   
1      1682                                  Mamma Roma (1962)   
2      1682                       Celluloid Closet, The (1995)   
3      1682                              Get on the Bus (1996)   
4      1682                                    Basquiat (1996)   
5      1682                                     Georgia (1995)   
6      1682                               Trainspotting (1996)   
7      1682                              Secrets & Lies (1996)   
8      1682  Last Klezmer: Leopold Kozlowski, His Life and ...   
9      1682                            Blue in the Face (1995)   

   predicted_rating  
0            4.8182  
1            4.8104  
2            4.8068  
3            4.8067  
4            4.8045  
5            4.7996  
6            4.7916  
7            4.7877  
8            4.7424  
9            4

### 9. Evaluate Performance
Measure performance of Item-Based CF using Root Mean Square Error (RMSE), Mean Absolute Error (MAE), and Execution Time (ms).


In [13]:
# Test set = 30%, Train set = 70%
actual = []
predicted = []

start_time = time.time()

for user_id in user_item_matrix.index:
    # Get the ratings of the target user
    user_ratings = user_item_matrix.loc[user_id]
    rated_movies = user_ratings[user_ratings > 0] # Rated movies only

    train_set = rated_movies.sample(frac=0.7) # 70% for train set
    test_set = rated_movies.drop(train_set.index) # Remaining 30% for test set

    for movie_id in test_set.index:
        true_rating = user_item_matrix.loc[user_id, movie_id]
        user_item_matrix.loc[user_id, movie_id] = 0 # Temporarily remove the rating for prediction

        prediction = predict_ratings(
            user_id,
            movie_id,
            user_item_matrix,
            item_similarity_df,
            N
        )

        # Restore the original rating
        user_item_matrix.loc[user_id, movie_id] = true_rating

        actual.append(true_rating)
        predicted.append(prediction)

end_time = time.time()

print("Root Mean Squared Error (RMSE):", round(sqrt(mean_squared_error(actual, predicted)), 4))
print("Mean Absolute Error (MAE):", round(mean_absolute_error(actual, predicted), 4))
print("Execution Time:", round(end_time - start_time, 4) * 1000, "ms")

Root Mean Squared Error (RMSE): 0.9578
Mean Absolute Error (MAE): 0.7434
Execution Time: 19691.8 ms
